In [ ]:
# ╔═══════════════════════════════════════════════╗
# ║  ★ 운영진 수정 구역 ★                        ║
# ╚═══════════════════════════════════════════════╝
PRED_START = "20260624"
PRED_END = "20260630"

# Kaggle Secrets 사용 금지: 최종 비공개 제출본에만 실제 키를 직접 입력
API_KEY = "REPLACE_WITH_SUBMISSION_API_KEY"

이 노트북은 추론 전용입니다. 평가 기간 ASOS/AWS 조회와 모델 재학습을 수행하지 않습니다. API 키가 들어간 최종 버전은 대회 종료 전에 공개하지 않습니다.

In [ ]:
import glob
import subprocess
import sys

REQUIREMENTS_PATH = glob.glob(
    "/kaggle/input/**/requirements-inference.txt", recursive=True
)[0]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", REQUIREMENTS_PATH]
)

In [ ]:
import joblib
import pandas as pd
import yaml

CODE_ROOT = glob.glob("/kaggle/input/**/gk2a_weather", recursive=True)[0]
sys.path.insert(0, CODE_ROOT.rsplit("/gk2a_weather", 1)[0])

from gk2a_weather.kaggle import run_kaggle_inference

MODEL_PATH = glob.glob("/kaggle/input/**/baseline.joblib", recursive=True)[0]
STATION_PATH = glob.glob("/kaggle/input/**/station_list.csv", recursive=True)[0]
CONFIG_PATH = glob.glob("/kaggle/input/**/data.yaml", recursive=True)[0]

bundle = joblib.load(MODEL_PATH)
stations = pd.read_csv(STATION_PATH)
with open(CONFIG_PATH, "r", encoding="utf-8") as stream:
    satellite_config = yaml.safe_load(stream)["satellite"]

pred_dates = pd.date_range(PRED_START, PRED_END, freq="D")
if len(pred_dates) == 0:
    raise ValueError("PRED_START는 PRED_END보다 늦을 수 없습니다.")

pred = run_kaggle_inference(
    api_key=API_KEY,
    pred_dates=pred_dates.tolist(),
    stations=stations.copy(),
    bundle=bundle,
    satellite_config=satellite_config,
    observation_hour_kst=14,
)

expected_rows = len(pred_dates) * len(stations)
assert len(pred) == expected_rows
assert pred[["TA", "HM"]].notna().all().all()
assert pred["HM"].between(0, 100).all()
pred.head()